Installed plotly to add annotation in plots for better readability and presentation

In [ ]:
# !pip install plotly
# pip install -U nbformat

Import Libraries such as numpy, pandas, matplotlib, seaborn, plotly etc as per requirement

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import iqr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px

Reading and analyzing data from Housing_data.csv

In [ ]:
#read data from excel file
df=pd.read_csv('C:/Users/ecsbb/OneDrive/Desktop/Sarita_AIML_Sessions/NextHikesInternship/Project3/housing_data.csv')
#drop unnecessary columns
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head()
#df.describe()

In [ ]:
df.info()
df.shape

Identifying columns with null values and managing them by droping the column or replacing the missing value.


In [ ]:
#identifying null
isnull=df.isnull().sum()
#list of columns having null values
ifnull=df.isnull().any()
null_columns=ifnull[ifnull==True].index.tolist()
print('Columns with null values:', isnull[null_columns])


In [ ]:
#identifying null values and getting columns with null values
#null_columns = df.columns[df.isnull().any()].tolist()

total_entries=len(df)

#Dropping columns having more than 60% null values and for the rest, replacing missing values with median/mode based on datatype
for column in null_columns:
    if column in df.columns:  # Check if column exists
        if isnull[column] / total_entries * 100 > 60:
            print(f"Column '{column}' has more than 60% null values: {isnull[column]} ({isnull[column]/total_entries*100:.2f}%)")
            df.drop(columns=[column], inplace=True)
            print(f'Deleted {column}')
        else:
            if df[column].dtype == 'object':
                df[column].fillna(df[column].mode()[0], inplace=True)
            else:
                df[column].fillna(df[column].median(), inplace=True)
# --- IGNORE ---
df.isnull().sum()

Finding Out duplicate values and handelling them. 

In [ ]:
df.duplicated().sum()

In [93]:
#Dropping duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df.shape

(1460, 79)

In [ ]:
#Catagorizing columns into numerical and non-numerical
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
Non_numerical_columns= df.select_dtypes(include=['object']).columns.tolist()

print("Numerical Columns:", num_cols)
print("Non-Numerical Columns:", Non_numerical_columns)

VISUALIZING THE DISTRIBUTION OF EACH UNIVARIATE COLUMNS [ITERATIVE]

In [ ]:
# Get all the columns and plot their distributions conditionally
# Iterate through columns
for col in df.columns:
    print(f'\n--- Analyzing Column: {col} ---')
    
    # Numerical data analysis
    if pd.api.types.is_numeric_dtype(df[col]): 
        print('Summary Stats (Numerical):')
        print(df[col].describe())
        
        # Visualization: Histogram (for distribution) & Boxplot (for outliers)
        # ---------- Plotly Histogram with Hover ----------
     
        fig_hist = px.histogram(df, x=col,nbins=20,title=f'Histogram of {col}',text_auto=True,
            #width=500, height=500,
            color_discrete_sequence=['indianred'],
            hover_data={col: True},
            opacity=0.75
        )
        fig_hist.update_traces(marker_line_width=1.5)
        fig_hist.update_layout(xaxis_title=col,yaxis_title='Frequency')
        fig_hist.show()

        #findout Skewness and Kurtosis
        skewness = df[col].skew()
        kurtosis = df[col].kurtosis()
        print(f'Skewness of {col}: {skewness}')
        print(f'Kurtosis of {col}: {kurtosis}') 
        if skewness > 1 or skewness < -1:
            print(f'{col} is Positively skewed.')
        elif 0.5 < skewness <= 1 or -1 <= skewness < -0.5:
            print(f'{col} is moderately skewed.')
        else:
            print(f'{col} is approximately symmetric.') 

        #Determine Outliers from Numerical Columns using scipy iqr
        col_IQR = iqr(df[col]) 
        lower_bound = df[col].quantile(0.25) - 1.5 * col_IQR
        upper_bound = df[col].quantile(0.75) + 1.5 * col_IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        print(f'Number of Outliers in {col}: {outliers.shape[0]}')   

        # ---------- Plotly Boxplot with Hover ----------
        fig_box = px.box(df, x=col,title=f'Boxplot of {col}',
            #width=500, height=500,
            points='outliers'                                       # show outliers interactively
        )

        fig_box.update_layout(xaxis_title=col) 
        fig_box.update_traces(marker_color='blue',line_color='black')  
        fig_box.show()

    else:
    # Categorical/Object data analysis with hover tooltips (Plotly)

        #Get Value Counts
        vc = df[col].value_counts().reset_index()
        vc.columns = [col, 'Count']

        # Plotly bar chart with hover
        fig = px.bar(vc,x=col,y='Count',text='Count',title=f'Countplot of {col}',
            color=col,color_discrete_sequence=px.colors.qualitative.Set2,
            hover_data={col: True, 'Count': True}
        )

        fig.update_traces(textposition='outside')                                           # Show count labels outside bars
        fig.update_layout(xaxis_title=col,yaxis_title='Frequency',xaxis_tickangle=-45)
        fig.show()
               
print('\n--- Univariate Analysis Complete ---')


OBSERVATION:
For Numerical Columns-
Features with 
1. low skewness, low kurtosis, and few outliers : LotFrontage, OverallQual, FullBath, BedroomAbvGr, GarageCars, GarageArea. 
                                                These demonstates symmtric districbution, so less processing required
2. Moderately Skewed Features (Noticiable yet managible): OverallCond,	YearBuilt,	YearRemodAdd,	BsmtUnfSF,	2ndFlrSF,	BsmtFullBath,	HalfBath,	TotRmsAbvGrd,	Fireplaces,	GarageYrBlt
3. High skewness, very high kurtosis, and many outliers : LotArea,	MasVnrArea,	BsmtFinSF1,	BsmtFinSF2,	TotalBsmtSF,	1stFlrSF,	LowQualFinSF,	GrLivArea,	BsmtHalfBath,	KitchenAbvGr,	    WoodDeckSF,	OpenPorchSF,	EnclosedPorch,	3SsnPorch,	ScreenPorch,	PoolArea,	MiscVal,	SalePrice,
                                                        These needs to be treated to achieve linear model.
Housing prices are primarily driven by living area, quality, and zoning/subclass effects

Most outliers occur in size and amenity based features such as lot size, basement areas, porches, pools, misc values
These represent meaningful market behavior rather than data errors so handled through transformation rather than removal.

For non-Numerical/ Object columns-
Majority of columns are imbanced due to dominant majority of category falling under the columns.
e.g. MSZoning with a dominant majority Residential Low Density (RL) zoning. It implies-
Homes in RL zones likely command higher and more stable prices due to larger lot sizes and lower density, while RM, RH, and C zones may exhibit different pricing dynamics driven by density and mixed-use constraints.
LotShape provides meaningful geometric information about the property. While regular lots dominate the dataset, varying degrees of irregularity can influence price and should be retained as an ordered categorical feature, with careful handling of rare extreme categories.
The dataset is dominated by a few residential neighborhoods, while many areas are sparsely represented, requiring careful encoding and potential grouping for robust modeling.

Determining and Handelling Outliers

In [ ]:
#Determine Outliers from Numerical Columns using scipy iqr and handle them
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_indices = {}    
for col in numerical_cols:
    col_IQR = iqr(df[col])
    lower_bound = df[col].quantile(0.25) - 1.5 * col_IQR
    upper_bound = df[col].quantile(0.75) + 1.5 * col_IQR
    outlier_indices[col] = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index.tolist()
    outliers = df.loc[outlier_indices[col]]
    if outliers.shape[0]<10:
        print(f'Number of Outliers in {col}: {outliers.shape[0]} (Displaying first 10 outliers)')
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)] 
        print('Outliers Removed')
    else:
        #replace outliers with median
        median_value = df[col].median()
        df.loc[outlier_indices[col], col] = median_value
        print(f'Replacing outliers in {col} with median value: {median_value}')
    print(f'Number of Outliers in {col}: {outliers.shape[0]}')

        
if any(len(indices) > 0 for indices in outlier_indices.values()):
    print('Outliers detected in the dataset.')   

Determining Corelation of each numeric column with target column 'SalePrice'

In [ ]:
#Finding Corelation of each numeric column with target column 'SalePrice'
correlation = df.select_dtypes(include=[np.number]).corr()['SalePrice'].sort_values(ascending=False)  
print("Correlation of each column with 'SalePrice':", correlation)

VISUALIZING THE BIVARIATE ANALYSIS OF EACH COLUMN AGAINST SALEPRICE COLUMN [ITERATIVE]

In [ ]:
# import plotly.io as pio
# pio.renderers.default = 'browser'                    #To render output on browser

# Get all the columns and plot them against SalePrice
# Iterate through columns
for col in df.columns:
    if col == 'SalePrice':
        continue

    print(f'\n--- Analyzing Column: {col} against SalePrice ---')

    # ---------- Numerical vs SalePrice ----------
    if pd.api.types.is_numeric_dtype(df[col]):
        print('Summary Stats (Numerical):')
        print(df[col].describe())
        
        # Visualization: Scatter plot for numerical vs SalePrice
        plt.figure(figsize=(20, 6))
        ax=sns.scatterplot(x=df[col], y=df['SalePrice'], hue=(df[col] > df[col].median()), palette='Set2',legend='auto')
        sns.regplot(data=df,x=df[col],y=df['SalePrice'],color='red',
                    scatter=False   # 🚫 don't redraw points
        )
        ax.legend(title=f'{col} > Median')                                  # Set legend title
        plt.xlabel(col)
        plt.ylabel('SalePrice')
        plt.title(f'Scatter plot of {col} vs SalePrice (Correlation: {correlation[col]:.3f})')
        plt.show()

    # ---------- Categorical vs SalePrice ----------
    else:
        fig = px.box(
            df,x=col,y='SalePrice',title=f'SalePrice by {col}',color=col,
            hover_data=[col, 'SalePrice'],
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.show()


OBSERVATION:(on few of the impacting columns)
Sale prices vary significantly across zoning categories, with FV and RL showing the highest median prices, while commercial zoning exhibits consistently lower values, indicating that MSZoning is a strong categorical predictor of housing prices.

Sale prices differ significantly across MSSubClass categories, with newer and multi-story dwelling types commanding higher prices, while older or conversion-style homes consistently sell for less.

SalePrice increases with YearBuilt showing a moderate positive correlation, but the wide dispersion indicates that these alone are insufficient to explain price without considering quality and other features.

OverallQual,Living area,GarageArea,GarageCars show a strong linear relationship with sale price, explaining a substantial portion of price variability. However, increasing variance at higher areas suggests the influence of additional quality and location factors.

BIVARIATE ANALYSIS - JOINT PLOT

In [ ]:
# Log Transformation of skewed features
skewed_cols = correlation[abs(correlation) > 0.5].index.tolist()  #columns having correlation more than 0.5 or less than -0.5
print('Skewed Columns:', skewed_cols)

# Log Transformation check using jointplot
JCols = ['GrLivArea','OverallQual','YearBuilt','TotalBsmtSF','GarageCars']
for col in JCols:
    sns.jointplot(data=df, x=np.log1p(df[col]), y=np.log1p(df['SalePrice']), height=6, ratio=3, color='g',hue='OverallQual')  #log1p is used to avoid log(0) issue
    
    #calculating slope and intercept for regression line
    slope, intercept = np.polyfit(np.log1p(df[col]), np.log1p(df['SalePrice']), 1)
    print(f'Slope: {slope}, Intercept: {intercept}')    
    
    plt.suptitle(f'{col} vs SalePrice', fontsize=16) 
    plt.show()

OBSERVATION:
A log transform helps:
    1. Linearize curved relationships
    2. Reduce the impact of outliers
    3.Stabilize variance
Slope is equivalant to elasticity. It signifies 1% increase in the logged columns → ~<slope>% increase in SalePrice
If slope were:
    ≈ 1 → almost proportional pricing (e.g. GrLivArea,Garage Car space. These are strong and consistent predictors of SalePrice)
    > 1 → luxury amplification effect (e.g. Overall Quality has an amplifying effect on sale price. This suggests premium valuation behavior rather than simple proportional pricing.)
    < 0.5 → weak or secondary feature (e.g. Total Basement that rarely influence price)
The extremely large elasticity/slope of YearBuilt indicates instability caused by feature scaling rather than a genuine economic effect, which implies Year Built is misleading feature for log transformation and can be ignored
While a linear relationship dominates, the presence of outliers and increasing price variance with size suggests the need for transformation or robust modeling techniques.

VISUALIZING THE MULTIVARIATE ANALYSIS

In [ ]:
#Correlation Matrix for Numerical Columns
corr_matrix = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(50, 25))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


Correlation Matrix interpretation
It ranges from -1 to +1.
A value close to +1 indicates a strong positive correlation, meaning as one variable increases, the other tends to increase as well.
A value close to -1 indicates a strong negative correlation, meaning as one variable increases, the other tends to decrease.    
A value close to 0 indicates little to no linear correlation between the variables.

More or less all the features has positive co-relation or impact on SalesPrice except YearSold, MiscVal (Value of Miscellaneous feature installed)

The relationship between key features like bedrooms, bathrooms, and square footage with house prices, determining their impact on valuation.

In [ ]:
cor_cols = df[['SalePrice','LotArea','OverallQual','TotalBsmtSF','BedroomAbvGr','GarageArea','GrLivArea','YearBuilt']]  # use only numeric columns for correlation
corr_matrix = cor_cols.corr()               #corr() is used to compute pairwise correlation of columns
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', annot_kws={'size': 10})  
plt.title('Correlation Matrix', fontsize=15)
plt.show()

MULTIVARIATE ANALYSIS - PAIR PLOT

In [ ]:
sns.pairplot(
    df[['SalePrice', 'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageArea','YearBuilt']],
    hue='OverallQual',diag_kind='kde',                                                       #hue means different colors for different categories
    height=4, aspect=0.8,kind='scatter',
    palette='Paired'
)
plt.show()

OBSERVATION:
The pair plot reveals strong positive relationships between GrLivArea, GarageArea, and SalePrice. Clear segmentation by OverallQual is visible, with higher quality homes consistently commanding higher prices across all size-related features. The distribution of SalePrice is right-skewed, indicating the presence of high-value outliers.

Price increases consistently with property size, while quality strongly amplifies valuation, making these features critical for predictive modeling.